# BuildersVault Social Services Hackathon, Track 2 Quickstart

**Food Security Delivery Operations**

Thirty-second description: this track gives you a realistic snapshot of a blended Meals on Wheels plus grocery-hamper delivery operation running in Victoria, BC. Your job is to turn messy, constraint-heavy delivery data into decisions that keep mobility-limited residents fed, safe, and on-schedule, without burning out the volunteers driving the vans.

## What the data represents

- **500 clients** in Victoria, all mobility-limited, with dietary, allergen, language, and accessibility profiles.
- **8 drivers** (6 volunteer, 2 paid staff) with skill tags (lift-trained, two-person-lift, first-aid, language fluency).
- **8 vehicles** (1 with wheelchair lift, 3 refrigerated, 4 standard) with capacity and cold-chain specs.
- **2 depots** (central kitchen + hamper warehouse) anchoring all routes.
- **300 routes** over 3 weeks with planned vs. actual stops, distances, durations, and on-time rates.
- **2,000 delivery requests** with time windows, program tags (MOW, hamper, cultural), and per-request dietary snapshots.
- **~10,000 route stops** with arrival times, failure reasons, and driver notes.
- **150 inventory items** tagged with allergens, dietary flags, cold-chain requirements.

## Four concrete challenges

1. **Route optimization**: shrink total drive time and maximize on-time rate while respecting time windows, vehicle capacity, and cold-chain.
2. **Mismatch detection**: surface wheelchair clients assigned to non-lift vehicles, allergen conflicts between delivered items and client profile, and language gaps where no driver-client shared language exists.
3. **Volunteer retention analytics**: predict no-show risk, flag early burnout signals, measure fairness of route assignment.
4. **Demand forecasting**: predict request volume by program type and depot to help the ops lead pre-stage inventory.

Pick one. Go deep. The rubric rewards depth over breadth.

In [ ]:
# Imports and config
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path('../data/raw')

if not DATA_DIR.exists():
    print(f"Data directory not found at {DATA_DIR.resolve()}")
    print("Run the data generation script from the track root first:")
    print("    python scripts/generate_data.py")
    sys.exit(0)

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Files present: {sorted(p.name for p in DATA_DIR.glob('*.parquet'))}")

In [ ]:
# Load all 9 Parquet files
clients = pd.read_parquet(DATA_DIR / 'clients.parquet')
drivers = pd.read_parquet(DATA_DIR / 'drivers.parquet')
vehicles = pd.read_parquet(DATA_DIR / 'vehicles.parquet')
depots = pd.read_parquet(DATA_DIR / 'depots.parquet')
routes = pd.read_parquet(DATA_DIR / 'routes.parquet')
requests = pd.read_parquet(DATA_DIR / 'delivery_requests.parquet')
stops = pd.read_parquet(DATA_DIR / 'route_stops.parquet')
items = pd.read_parquet(DATA_DIR / 'inventory_items.parquet')
request_items = pd.read_parquet(DATA_DIR / 'delivery_request_items.parquet')

tables = {
    'clients': clients,
    'drivers': drivers,
    'vehicles': vehicles,
    'depots': depots,
    'routes': routes,
    'delivery_requests': requests,
    'route_stops': stops,
    'inventory_items': items,
    'delivery_request_items': request_items,
}

for name, df in tables.items():
    print(f"{name:28s} shape={df.shape}")

In [ ]:
# .info() and null summary across all 9 tables
summary_rows = []
for name, df in tables.items():
    nulls = df.isna().sum()
    pct_nulls = (nulls / len(df) * 100).round(2)
    null_cols = pct_nulls[pct_nulls > 0]
    summary_rows.append({
        'table': name,
        'rows': len(df),
        'cols': df.shape[1],
        'columns_with_nulls': len(null_cols),
        'max_null_pct': float(null_cols.max()) if len(null_cols) else 0.0,
        'worst_null_column': null_cols.idxmax() if len(null_cols) else None,
    })

null_summary = pd.DataFrame(summary_rows)
print(null_summary.to_string(index=False))

In [ ]:
# Schema validation, per table
expected_schemas = {
    'clients': {
        'client_id': 'object', 'full_name': 'object', 'age': 'int64',
        'mobility_level': 'object', 'wheelchair_user': 'bool',
        'requires_two_person': 'bool', 'primary_language': 'object',
        'home_depot': 'object', 'food_security_level': 'object',
        'enrolment_status': 'object',
    },
    'drivers': {
        'driver_id': 'object', 'full_name': 'object', 'is_volunteer': 'bool',
        'lift_trained': 'bool', 'two_person_trained': 'bool',
        'languages': 'object', 'home_depot': 'object',
        'max_hours_per_week': 'float64',
    },
    'vehicles': {
        'vehicle_id': 'object', 'has_lift': 'bool', 'refrigerated': 'bool',
        'capacity_kg': 'float64', 'home_depot': 'object',
    },
    'depots': {
        'depot_id': 'object', 'name': 'object',
        'latitude': 'float64', 'longitude': 'float64',
    },
    'routes': {
        'route_id': 'object', 'route_date': 'object', 'driver_id': 'object',
        'vehicle_id': 'object', 'depot_id': 'object',
        'planned_stops': 'int64', 'completed_stops': 'int64',
        'on_time_rate': 'float64', 'status': 'object',
    },
    'delivery_requests': {
        'request_id': 'object', 'client_id': 'object',
        'program_type': 'object', 'window_start': 'object',
        'window_end': 'object', 'priority': 'object', 'notes': 'object',
    },
    'route_stops': {
        'stop_id': 'object', 'route_id': 'object', 'request_id': 'object',
        'sequence': 'int64', 'arrival_time': 'object',
        'status': 'object', 'failure_reason': 'object',
    },
    'inventory_items': {
        'item_id': 'object', 'name': 'object', 'allergens': 'object',
        'dietary_flags': 'object', 'cold_chain': 'bool',
    },
    'delivery_request_items': {
        'request_id': 'object', 'item_id': 'object', 'quantity': 'int64',
    },
}

results = []
for name, expected in expected_schemas.items():
    df = tables[name]
    missing = [c for c in expected if c not in df.columns]
    wrong_dtype = []
    for col, dtype in expected.items():
        if col in df.columns and not str(df[col].dtype).startswith(dtype.split('64')[0]):
            wrong_dtype.append(f"{col}({df[col].dtype} vs {dtype})")
    passed = not missing and not wrong_dtype
    results.append({
        'table': name,
        'status': 'PASS' if passed else 'FAIL',
        'missing': ','.join(missing) or '-',
        'wrong_dtype': ','.join(wrong_dtype) or '-',
    })

schema_report = pd.DataFrame(results)
print(schema_report.to_string(index=False))
assert (schema_report['status'] == 'PASS').all(), "Schema validation failed, see report above."
print("\nAll 9 tables passed schema validation.")

In [ ]:
# Golden join, stops_enriched = stops + requests + clients + routes + drivers + item count
items_per_request = (
    request_items.groupby('request_id')['quantity'].sum().rename('items_delivered_count')
)

stops_enriched = (
    stops
    .merge(requests, on='request_id', how='left', suffixes=('', '_req'))
    .merge(clients, on='client_id', how='left', suffixes=('', '_client'))
    .merge(routes, on='route_id', how='left', suffixes=('', '_route'))
    .merge(drivers, on='driver_id', how='left', suffixes=('', '_driver'))
    .merge(items_per_request, on='request_id', how='left')
)

key_cols = [
    'stop_id', 'route_id', 'sequence', 'client_id', 'full_name',
    'program_type', 'window_start', 'window_end', 'arrival_time',
    'status', 'failure_reason', 'wheelchair_user', 'requires_two_person',
    'driver_id', 'is_volunteer', 'items_delivered_count',
]
key_cols = [c for c in key_cols if c in stops_enriched.columns]
print(f"stops_enriched shape: {stops_enriched.shape}")
stops_enriched[key_cols].head(10)

In [ ]:
# EDA, 2x3 grid of 6 plots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Route status breakdown
routes['status'].value_counts().plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Route Status Breakdown')
axes[0, 0].set_xlabel('status')
axes[0, 0].set_ylabel('count')
axes[0, 0].tick_params(axis='x', rotation=30)

# 2. On-time rate distribution across routes
routes['on_time_rate'].dropna().plot(kind='hist', bins=25, ax=axes[0, 1], color='seagreen', edgecolor='white')
axes[0, 1].set_title('On-Time Rate Distribution Across Routes')
axes[0, 1].set_xlabel('on_time_rate')
axes[0, 1].set_ylabel('route count')

# 3. Request volume by program type
requests['program_type'].value_counts().plot(kind='bar', ax=axes[0, 2], color='darkorange')
axes[0, 2].set_title('Request Volume by Program Type')
axes[0, 2].set_xlabel('program_type')
axes[0, 2].set_ylabel('requests')
axes[0, 2].tick_params(axis='x', rotation=30)

# 4. Client food security level distribution
clients['food_security_level'].value_counts().plot(kind='bar', ax=axes[1, 0], color='purple')
axes[1, 0].set_title('Client Food Security Level')
axes[1, 0].set_xlabel('food_security_level')
axes[1, 0].set_ylabel('client count')
axes[1, 0].tick_params(axis='x', rotation=30)

# 5. No-answer rate per driver, ranked
driver_stops = stops_enriched.groupby('driver_id').agg(
    total=('stop_id', 'count'),
    no_answer=('failure_reason', lambda s: (s == 'no_answer').sum()),
)
driver_stops['no_answer_rate'] = (driver_stops['no_answer'] / driver_stops['total']).fillna(0)
driver_stops = driver_stops.sort_values('no_answer_rate', ascending=False)
driver_stops['no_answer_rate'].plot(kind='bar', ax=axes[1, 1], color='crimson')
axes[1, 1].set_title('No-Answer Rate per Driver')
axes[1, 1].set_xlabel('driver_id')
axes[1, 1].set_ylabel('no_answer_rate')
axes[1, 1].tick_params(axis='x', rotation=60)

# 6. Stops per hour over the day (8..18)
arrival = pd.to_datetime(stops['arrival_time'], errors='coerce')
hour_counts = arrival.dt.hour.value_counts().sort_index()
hours = list(range(8, 19))
hour_series = pd.Series([hour_counts.get(h, 0) for h in hours], index=hours)
hour_series.plot(ax=axes[1, 2], marker='o', color='teal')
axes[1, 2].set_title('Stops per Hour of Day')
axes[1, 2].set_xlabel('hour')
axes[1, 2].set_ylabel('stops')
axes[1, 2].set_xticks(hours)

plt.tight_layout()
plt.show()

In [ ]:
# Baseline model, greedy route quality scorer
# quality_score = 0.4 * on_time_rate + 0.3 * completion_rate + 0.2 * skill_match_rate - 0.1 * cold_chain_violations_norm

route_stops_grp = stops_enriched.groupby('route_id')

completion = route_stops_grp.apply(
    lambda g: (g['status'] == 'completed').sum() / max(len(g), 1)
).rename('completion_rate')

def _skill_match(g):
    if len(g) == 0:
        return 1.0
    wc_clients = g['wheelchair_user'].fillna(False)
    # skill mismatch proxy, wheelchair client on a route with no lift-trained driver
    lift = g['lift_trained'].fillna(False) if 'lift_trained' in g.columns else pd.Series([False] * len(g))
    needs_lift = wc_clients
    matched = (~needs_lift) | lift
    return matched.sum() / len(g)

skill_match = route_stops_grp.apply(_skill_match).rename('skill_match_rate')

cold_violations = route_stops_grp.apply(
    lambda g: (g['failure_reason'] == 'cold_chain_violation').sum()
).rename('cold_chain_violations')
max_cv = max(cold_violations.max(), 1)
cold_violations_norm = (cold_violations / max_cv).rename('cold_chain_violations_norm')

score_df = (
    routes[['route_id', 'driver_id', 'on_time_rate', 'planned_stops', 'completed_stops']]
    .merge(completion, on='route_id', how='left')
    .merge(skill_match, on='route_id', how='left')
    .merge(cold_violations, on='route_id', how='left')
    .merge(cold_violations_norm, on='route_id', how='left')
    .fillna({'on_time_rate': 0, 'completion_rate': 0, 'skill_match_rate': 1,
             'cold_chain_violations': 0, 'cold_chain_violations_norm': 0})
)

score_df['quality_score'] = (
    0.4 * score_df['on_time_rate']
    + 0.3 * score_df['completion_rate']
    + 0.2 * score_df['skill_match_rate']
    - 0.1 * score_df['cold_chain_violations_norm']
)

# Naive nearest-neighbour distance delta, placeholder heuristic
# (assume planned route distance vs sum of sequential stop counts, as a stand-in)
score_df['nn_savings_estimate_pct'] = np.where(
    score_df['planned_stops'] > 0,
    (1 - score_df['quality_score']) * 15.0,  # up to ~15% theoretical savings on worst routes
    0,
)

worst = score_df.sort_values('quality_score').head(5)
print("Top 5 worst-routed routes:")
print(worst[['route_id', 'driver_id', 'on_time_rate', 'completion_rate',
             'skill_match_rate', 'cold_chain_violations', 'quality_score',
             'nn_savings_estimate_pct']].to_string(index=False))

In [ ]:
# Viz summary, two plots
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# (a) Client locations coloured by home_depot, depot pins highlighted
if {'latitude', 'longitude', 'home_depot'}.issubset(clients.columns):
    for depot_id, sub in clients.groupby('home_depot'):
        axes[0].scatter(sub['longitude'], sub['latitude'], s=12, alpha=0.5, label=f'depot {depot_id}')
    if {'latitude', 'longitude'}.issubset(depots.columns):
        axes[0].scatter(depots['longitude'], depots['latitude'], s=220, c='black',
                        marker='*', edgecolor='gold', linewidth=1.5, label='depot', zorder=5)
    axes[0].set_title('Client Locations by Home Depot (Victoria)')
    axes[0].set_xlabel('longitude')
    axes[0].set_ylabel('latitude')
    axes[0].legend(loc='best', fontsize=8)
else:
    axes[0].text(0.5, 0.5, 'client lat/lon not available', ha='center', va='center')
    axes[0].set_axis_off()

# (b) Top 10 mismatch incidents, horizontal bar
mismatch_counts = {}
if 'wheelchair_user' in stops_enriched.columns and 'has_lift' in vehicles.columns:
    veh_lift = vehicles.set_index('vehicle_id')['has_lift']
    route_veh = routes.set_index('route_id')['vehicle_id']
    stop_lift = stops_enriched['route_id'].map(route_veh).map(veh_lift).fillna(False)
    wc_no_lift = (stops_enriched['wheelchair_user'].fillna(False)) & (~stop_lift)
    mismatch_counts['wheelchair client, no-lift vehicle'] = int(wc_no_lift.sum())

if 'failure_reason' in stops_enriched.columns:
    fr = stops_enriched['failure_reason'].fillna('')
    mismatch_counts['allergen conflict flagged'] = int((fr == 'allergen_conflict').sum())
    mismatch_counts['interpreter gap flagged'] = int((fr == 'language_gap').sum())
    mismatch_counts['dog at address, allergic driver'] = int((fr == 'pet_conflict').sum())
    mismatch_counts['two-person unavailable'] = int((fr == 'requires_two_person_unavailable').sum())
    mismatch_counts['post-closure attempt'] = int((fr == 'post_closure_attempt').sum())
    mismatch_counts['cold chain violation'] = int((fr == 'cold_chain_violation').sum())
    mismatch_counts['address not found'] = int((fr == 'address_not_found').sum())
    mismatch_counts['refused delivery'] = int((fr == 'refused').sum())
    mismatch_counts['no answer'] = int((fr == 'no_answer').sum())

mismatch_series = pd.Series(mismatch_counts).sort_values(ascending=True).tail(10)
mismatch_series.plot(kind='barh', ax=axes[1], color='tomato')
axes[1].set_title('Top 10 Mismatch / Failure Incidents')
axes[1].set_xlabel('count')

plt.tight_layout()
plt.show()

## Extension ideas and rubric alignment

Pick one angle and go deep. The judges reward shipped demos over slide decks.

| Extension | Rough approach | Maps to rubric theme |
|---|---|---|
| **OR-Tools VRP optimizer** | Google OR-Tools `pywrapcp`, capacity + time-window + skill dimensions, minimize total travel + tardiness. Open-source, free. | Technical Merit |
| **Graph-based compatibility matching** | Build a bipartite graph of (client needs) vs (driver + vehicle capabilities). Score each edge. Use for assignment + to explain why a given pairing is risky. | Social Services Domain Grounding |
| **No-answer / no-show prediction** | Per-client time-series of failed deliveries, simple gradient boosting on day-of-week, window-length, weather, prior failure rate. | Problem Fit |
| **Demand forecasting by program + season** | Group by `program_type` + depot + week, fit Prophet or simple ETS. Show pre-staging recommendation. | Problem Fit |
| **Volunteer scheduling with fairness constraints** | Linear program: minimize driver hours variance, respect max-hours and max-distance, keep volunteers away from their hardest stops two weeks in a row. | Social Services Domain Grounding |
| **Cognitive-load-lite operator UI** | Streamlit or Next.js page with one clear "today's anomalies" panel. No dashboards-of-dashboards. | Production Readiness |
| **Allergen + dietary safety checker** | Pre-delivery validator: for each `(request_id, item_id)` pair, hard-stop if client allergen severe matches item allergen. Output audit log. | Social Services Domain Grounding |
| **What-if replay** | Let operator swap a driver or vehicle and re-score the route. Useful on sick days. | Production Readiness |

### Judging themes, plain-english version

- **Problem Fit**: does your build actually help a volunteer ops lead tomorrow morning?
- **Technical Merit**: clean data flow, defensible models, no magic numbers.
- **Social Services Domain Grounding**: respects consent, safety, dignity. No "optimize the poor" vibes.
- **Production Readiness**: a demo a non-technical operator can click through without your help.

Good luck. Ping the BuildersVault Slack if you hit a data issue.